# Пайплайн обучения, телеметрии и генерации модели D16Qwen с нуля

Этот блокнот реализует полный цикл работы с архитектурой `D16QwenForCausalLM` (`src/d16_qwen_model.py`) на базе объектов `D16ComplexTensor` (`src/d16_complex.py`) без использования внешних предобученных весов:
1. **Подготовка токенизатора и обучающей выборки**.
2. **Инициализация модели `D16QwenForCausalLM`** в режимах `d16_swiglu` (совместимом с Qwen2.5) и `d16_bipolar` (полном двухкомпонентном режиме $\mathrm{Re}(z^2), \mathrm{Im}(z^2)$ с 18-секторным профилем усиления).
3. **Цикл обучения с нуля** с прохождением градиентов через фундаментальный клин $[0^\circ, 22.5^\circ]$ (STE).
4. **Проверка авторегрессионной генерации** и эквивалентности `KV-cache`.
5. **Снятие карты 18 состояний `D16Complex`**, проверка разреженности состояния `16: Indifferent` (`0xFFFF`) и побитового управления маской сектора (`set_sector_steering`).

In [ ]:
import math
from pathlib import Path
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

# Подключаем папку src из корня проекта D:\repo\d16nn\qwen\src
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "ipynb" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from d16_complex import D16Complex, D16ComplexTensor, STATE_LABELS
from d16_qwen_model import D16QwenConfig, D16QwenForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print(f"Using device: {device}")
print(f"Loaded modules from: {SRC_DIR}")

## 1. Подготовка словаря, токенизатора и обучающей выборки
Создадим компактный пословный/символьный токенизатор и набор обучающих текстов (вопросы-ответы и определения), чтобы наглядно проверить сходимость и осмысленную генерацию с нуля.

In [ ]:
class SimpleWordTokenizer:
    """Легковесный токенизатор без внешних зависимостей для проверки пайплайна с нуля."""

    def __init__(self, texts):
        special = ["<pad>", "<bos>", "<eos>", "<unk>"]
        words = sorted({tok for text in texts for tok in self._split(text)})
        self.itos = special + words
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id = 0
        self.bos_id = 1
        self.eos_id = 2
        self.unk_id = 3

    @staticmethod
    def _split(text: str):
        for ch in ["?", ".", ",", ":", ";"]:
            text = text.replace(ch, f" {ch} ")
        return text.strip().split()

    @property
    def vocab_size(self) -> int:
        return len(self.itos)

    def encode(self, text: str, add_bos: bool = True, add_eos: bool = True) -> list[int]:
        ids = [self.stoi.get(t, self.unk_id) for t in self._split(text)]
        if add_bos:
            ids = [self.bos_id] + ids
        if add_eos:
            ids = ids + [self.eos_id]
        return ids

    def decode(self, ids) -> str:
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        tokens = [
            self.itos[i] for i in ids
            if 0 <= i < len(self.itos) and self.itos[i] not in ("<pad>", "<bos>", "<eos>")
        ]
        text = " ".join(tokens)
        for ch in ["?", ".", ",", ":", ";"]:
            text = text.replace(f" {ch}", ch)
        return text


TRAIN_CORPUS = [
    "Q: Does the sun rise in the east? A: Yes, the sun rises in the east and sets in the west.",
    "Q: Is water wet and liquid? A: Yes, water is a clear liquid essential for life.",
    "Q: Is fire cold like ice? A: No, fire is hot and ice is cold.",
    "Q: What is D16Complex? A: D16Complex encodes vectors into the fundamental wedge and a 16-bit sector mask.",
    "Q: How does sector steering work? A: Sector steering rotates only the 16-bit mask without changing wedge coordinates.",
    "Review: The book is inspiring and captivating. Summary: This book is a wonderful masterpiece.",
    "Review: The movie was boring and poorly written. Summary: This movie is a terrible disaster.",
    "Вопрос: Вращается ли Земля вокруг Солнца? Ответ: Да, Земля движется по орбите вокруг Солнца.",
    "Вопрос: Как работает нейрон D16? Ответ: Нейрон вычисляет комплексный квадрат внутри клина двадцать два с половиной градуса.",
]

tokenizer = SimpleWordTokenizer(TRAIN_CORPUS)
encoded_seqs = [torch.tensor(tokenizer.encode(t), dtype=torch.long) for t in TRAIN_CORPUS]
max_len = max(len(s) for s in encoded_seqs)

input_batch = torch.full((len(encoded_seqs), max_len), tokenizer.pad_id, dtype=torch.long)
attn_mask = torch.zeros((len(encoded_seqs), max_len), dtype=torch.long)
for i, seq in enumerate(encoded_seqs):
    input_batch[i, :len(seq)] = seq
    attn_mask[i, :len(seq)] = 1

input_batch = input_batch.to(device)
attn_mask = attn_mask.to(device)

print(f"Размер словаря: {tokenizer.vocab_size} токенов")
print(f"Форма обучающего батча: {tuple(input_batch.shape)}")

## 2. Создание объекта модели `D16QwenForCausalLM`
Создадим две конфигурации сети:
- `d16_swiglu` — 1-к-1 режим Qwen2.5, где нелинейность вычисляется как $\frac{1}{2}\mathrm{Im}(z^2)\sigma(\mathrm{Im}(z))$ через квадрат в клине $[0^\circ, 22.5^\circ]$.
- `d16_bipolar` — нативный двухкомпонентный режим, использующий как $\mathrm{Re}(z^2)$, так и $\mathrm{Im}(z^2)$ вместе с обучаемыми весами 18 состояний `sector_gain`.

In [ ]:
def create_d16_model(mlp_mode: str = "d16_swiglu") -> D16QwenForCausalLM:
    cfg = D16QwenConfig(
        vocab_size=tokenizer.vocab_size,
        hidden_size=128,
        intermediate_size=256,
        num_hidden_layers=4,
        num_attention_heads=4,
        num_key_value_heads=2,
        max_position_embeddings=512,
        rms_norm_eps=1e-6,
        rope_theta=10000.0,
        tie_word_embeddings=True,
        mlp_mode=mlp_mode,
        indiff_eps=0.0,
    )
    return D16QwenForCausalLM(cfg).to(device)


model_swiglu = create_d16_model("d16_swiglu")
model_bipolar = create_d16_model("d16_bipolar")

params_swiglu = sum(p.numel() for p in model_swiglu.parameters())
params_bipolar = sum(p.numel() for p in model_bipolar.parameters())
print(f"D16Qwen (d16_swiglu)  параметров: {params_swiglu:,d}")
print(f"D16Qwen (d16_bipolar) параметров: {params_bipolar:,d}")

## 3. Запуск цикла обучения с нуля
Обучим обе модели на подготовленном корпусе с маскированием `<pad>`-токенов и сравним динамику `Loss` и `Perplexity`.

In [ ]:
def train_model(model: D16QwenForCausalLM, steps: int = 120, lr: float = 5e-3):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=1e-4)

    # Формируем целевые метки со значением -100 на позициях <pad>
    labels = input_batch.clone()
    labels[input_batch == tokenizer.pad_id] = -100

    history = []
    for step in range(1, steps + 1):
        optimizer.zero_grad()
        out = model(input_ids=input_batch, attention_mask=attn_mask)
        shift_logits = out.logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, model.config.vocab_size),
            shift_labels.view(-1),
            ignore_index=-100,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        loss_val = float(loss.item())
        ppl_val = math.exp(min(loss_val, 20.0))
        history.append((step, loss_val, ppl_val))
        if step == 1 or step % 20 == 0 or step == steps:
            print(f"  Шаг {step:3d}/{steps} | NLL Loss: {loss_val:.4f} | Perplexity: {ppl_val:.2f}")
    return history


print("=== Обучение D16Qwen (режим d16_swiglu) ===")
hist_swiglu = train_model(model_swiglu, steps=120, lr=5e-3)

print("\n=== Обучение D16Qwen (режим d16_bipolar) ===")
hist_bipolar = train_model(model_bipolar, steps=120, lr=5e-3)

## 4. Проверка генерации и работы `KV-cache`
Проверим авторегрессионную генерацию ответов по префиксам вопросов и убедимся, что генерация с `use_cache=True` полностью совпадает с `use_cache=False`.

In [ ]:
test_prompts = [
    "Q: Does the sun rise in the east? A:",
    "Q: Is fire cold like ice? A:",
    "Q: What is D16Complex? A:",
    "Review: The book is inspiring and captivating. Summary:",
    "Вопрос: Как работает нейрон D16? Ответ:",
]

for name, mdl in [("d16_swiglu", model_swiglu), ("d16_bipolar", model_bipolar)]:
    print("=" * 92)
    print(f"ГЕНЕРАЦИЯ МОДЕЛИ: {name}")
    print("=" * 92)
    mdl.eval()
    for prompt in test_prompts:
        inp = torch.tensor([tokenizer.encode(prompt, add_bos=True, add_eos=False)], dtype=torch.long, device=device)
        out_cached = mdl.generate(inp, max_new_tokens=16, do_sample=False, eos_token_id=tokenizer.eos_id, use_cache=True)
        out_nocache = mdl.generate(inp, max_new_tokens=16, do_sample=False, eos_token_id=tokenizer.eos_id, use_cache=False)
        assert torch.equal(out_cached, out_nocache), "Несовпадение между KV-cache и полным проходом!"
        print(f"  Промпт : {prompt}")
        print(f"  Ответ  : {tokenizer.decode(out_cached[0])}")
        print("-" * 92)

## 5. Телеметрия 18 состояний `D16Complex`, порог `0xFFFF` (`Indifferent`) и управление маской сектора
Соберём статистику распределения активаций по 16 угловым секторам $[k\cdot 22.5^\circ, (k+1)\cdot 22.5^\circ)$ и состоянию `16: Indifferent` (`0xFFFF`), а затем проверим влияние порога `indiff_eps` и побитового сдвига маски сектора на живую генерацию.

In [ ]:
@torch.no_grad()
def evaluate_corpus_loss(model: D16QwenForCausalLM) -> tuple[float, float]:
    model.eval()
    labels = input_batch.clone()
    labels[input_batch == tokenizer.pad_id] = -100
    out = model(input_ids=input_batch, attention_mask=attn_mask)
    shift_logits = out.logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    loss = F.cross_entropy(
        shift_logits.view(-1, model.config.vocab_size),
        shift_labels.view(-1),
        ignore_index=-100,
    )
    nll = float(loss.item())
    return nll, math.exp(min(nll, 20.0))


# 1. Карта 18 состояний D16Complex при eps = 0.0
model_swiglu.set_indifferent_threshold(0.0)
model_swiglu.set_collect_d16_stats(True, reset=True)
_ = evaluate_corpus_loss(model_swiglu)
summary = model_swiglu.get_d16_state_summary()
counts = summary["state_counts"]
total_el = max(summary["total_elements"], 1)

print("=" * 78)
print("КАРТА 18 СОСТОЯНИЙ D16Complex ПО ВСЕМ СЛОЯМ МОДЕЛИ")
print("=" * 78)
for s_idx in range(18):
    cnt = int(counts[s_idx].item())
    pct = cnt / total_el * 100.0
    print(f"  {s_idx:2d}: {STATE_LABELS[s_idx]:<14} | {cnt:8,d} срабатываний | {pct:6.2f}%")

# 2. Проверка порога покоя 16: Indifferent (0xFFFF)
print("\n" + "=" * 78)
print("ВЛИЯНИЕ ПОРОГА 16:Indifferent (0xFFFF) НА РАЗРЕЖЕННОСТЬ И NLL")
print("=" * 78)
for eps in (0.0, 0.05, 0.15, 0.25, 0.35):
    model_swiglu.set_indifferent_threshold(eps)
    model_swiglu.set_collect_d16_stats(True, reset=True)
    nll, ppl = evaluate_corpus_loss(model_swiglu)
    sp = model_swiglu.get_d16_state_summary()["indifferent_pct"]
    print(f"  eps = {eps:.2f} | Доля 0xFFFF: {sp:6.2f}% | NLL: {nll:.4f} | PPL: {ppl:.2f}")

model_swiglu.set_indifferent_threshold(0.0)
model_swiglu.set_collect_d16_stats(False)

# 3. Побитовое управление маской сектора на последнем слое (координаты x, y в клине неизменны!)
print("\n" + "=" * 78)
print("ПОБИТОВОЕ УПРАВЛЕНИЕ МАСКОЙ СЕКТОРА (КООРДИНАТЫ (x, y) В [0°, 22.5°] НЕИЗМЕННЫ)")
print("=" * 78)
prompt = "Review: The book is inspiring and captivating. Summary:"
inp = torch.tensor([tokenizer.encode(prompt, add_bos=True, add_eos=False)], dtype=torch.long, device=device)

for mode, shift, desc in [
    ("identity", 0, "Исходный сектор (identity)"),
    ("active_swap_le", 0, "Отражение 45° в активных секторах 0..7 (active_swap_le)"),
    ("active_not_logic", 0, "Отражение оси L в активных секторах 0..7 (active_not_logic)"),
    ("rotate", 4, "Поворот маски +90° (rol 4)"),
]:
    model_swiglu.set_sector_steering(mode=mode, shift=shift, target_layers=[3])
    out_ids = model_swiglu.generate(inp, max_new_tokens=10, do_sample=False, eos_token_id=tokenizer.eos_id)
    print(f"  {desc:<58} -> {tokenizer.decode(out_ids[0])}")

model_swiglu.set_sector_steering(mode="identity")